# Module 2

In [1]:
# import AWS SDK for Python
import boto3

# import json and sys
import json
import sys

## Listing Foundation Models

In [2]:

# Instantiate a bedrock client
bedrock = boto3.client("bedrock", region_name="us-east-1")
# List foundation models
response = bedrock.list_foundation_models()

# print it the answer in a pretty way
print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "72924d1d-4f54-4cfa-b478-d15dcbff5321",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 30 Jun 2026 17:21:20 GMT",
      "content-type": "application/json",
      "content-length": "203380",
      "connection": "keep-alive",
      "x-amzn-requestid": "72924d1d-4f54-4cfa-b478-d15dcbff5321"
    },
    "RetryAttempts": 0
  },
  "modelSummaries": [
    {
      "modelArn": "arn:aws:bedrock:us-east-1::foundation-model/nvidia.nemotron-nano-12b-v2",
      "modelId": "nvidia.nemotron-nano-12b-v2",
      "modelName": "NVIDIA Nemotron Nano 12B v2 VL BF16",
      "providerName": "NVIDIA",
      "inputModalities": [
        "TEXT",
        "IMAGE"
      ],
      "outputModalities": [
        "TEXT"
      ],
      "responseStreamingSupported": true,
      "customizationsSupported": [],
      "inferenceTypesSupported": [
        "ON_DEMAND"
      ],
      "modelLifecycle": {
        "status": "ACTIVE"
      }
    },
    {
      "modelArn"

## Invoking models using invokeModel

In [3]:

# Instantiate a bedrock-runtime client in us-east-1
bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

### Llama 3's instruct model

In [4]:
# Embed the prompt in Llama 3's instruction format.
prompt = "Describe the purpose of a 'hello world' program in one line."
formatted_prompt = f"""
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{prompt}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

# Invoke the model
response = bedrock_runtime.invoke_model(
    modelId="meta.llama3-70b-instruct-v1:0", 
    body=json.dumps({
        "prompt": formatted_prompt,
        "max_gen_len": 512,
        "temperature": 0.5,
    })
)

# Decode the response body and print it
model_response = json.loads(response["body"].read())
print(json.dumps(model_response, indent=2))

{
  "generation": "A \"Hello, World!\" program is a traditional introductory program that serves as a sanity check to verify that a programming language, compiler, and development environment are correctly installed and functioning properly.",
  "prompt_token_count": 26,
  "generation_token_count": 38,
  "stop_reason": "stop"
}


### Mistral AI

In [5]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."
formatted_prompt = f"<s>[INST]{prompt}[/INST]"

# Invoke the model
response = bedrock_runtime.invoke_model(
    modelId="mistral.mistral-large-2402-v1:0", 
    body=json.dumps({
        "prompt": formatted_prompt,
        "max_tokens": 512,
        "temperature": 0.5,
    })
)

# Decode the response body and print it
model_response = json.loads(response["body"].read())
print(json.dumps(model_response, indent=2))

{
  "outputs": [
    {
      "text": " The purpose of a 'hello world' program is to provide a simple introduction to a programming language, demonstrating its basic syntax and confirming that the development environment is correctly set up.",
      "stop_reason": "stop"
    }
  ]
}


### Amazon Nova

In [9]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Invoke the model
response = bedrock_runtime.invoke_model(
    modelId="amazon.nova-lite-v1:0", 
    body=json.dumps({
        "inferenceConfig": {
            "maxTokens": 512,
            "temperature": 0.5
        },
        "messages": [
            {
                "role": "user",
                "content": [{
                    "text": prompt
                }]
            }
        ]
    })
)

# Decode the response body and print it
model_response = json.loads(response["body"].read())
print(json.dumps(model_response, indent=2))

{
  "output": {
    "message": {
      "content": [
        {
          "text": "The purpose of a 'hello world' program is to provide a simple introduction to the basic syntax of a programming language."
        }
      ],
      "role": "assistant"
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 14,
    "outputTokens": 25,
    "totalTokens": 39,
    "cacheReadInputTokenCount": 0,
    "cacheWriteInputTokenCount": 0
  }
}


## Converse API

### Llama 3's instruct model

In [6]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Invoke the model using converse
response = bedrock_runtime.converse(
    modelId="meta.llama3-70b-instruct-v1:0", 
    inferenceConfig = {
        "maxTokens": 512,
        "temperature": 0.5
    },
    messages = [
        {
            "role": "user",
            "content": [{
                "text": prompt
            }]
        }
    ]
)

# Print the response output
print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "613641a1-d910-4061-a898-3bf5ae88128f",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 30 Jun 2026 17:22:38 GMT",
      "content-type": "application/json",
      "content-length": "432",
      "connection": "keep-alive",
      "x-amzn-requestid": "613641a1-d910-4061-a898-3bf5ae88128f"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "\n\nA \"hello world\" program is a traditional introductory exercise in programming that serves as a minimal test to verify that a language's compiler, development environment, and runtime are correctly installed and functioning."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 28,
    "outputTokens": 40,
    "totalTokens": 68
  },
  "metrics": {
    "latencyMs": 1078
  }
}


### Mistral AI

In [11]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Invoke the model using converse
response = bedrock_runtime.converse(
    modelId="mistral.mistral-large-2402-v1:0", 
    inferenceConfig = {
        "maxTokens": 512,
        "temperature": 0.5
    },
    messages = [
        {
            "role": "user",
            "content": [{
                "text": prompt
            }]
        }
    ]
)

# Print the response output
print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "7fda9656-28d9-468f-87ec-dc0b5c4341b4",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 29 Jun 2026 15:27:11 GMT",
      "content-type": "application/json",
      "content-length": "404",
      "connection": "keep-alive",
      "x-amzn-requestid": "7fda9656-28d9-468f-87ec-dc0b5c4341b4"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "The purpose of a 'hello world' program is to provide a simple introduction to a programming language, demonstrating its basic syntax and confirming that the development environment is working correctly."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 17,
    "outputTokens": 37,
    "totalTokens": 54
  },
  "metrics": {
    "latencyMs": 880
  }
}


### Amazon Nova

In [13]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Invoke the model using converse
response = bedrock_runtime.converse(
    modelId="us.amazon.nova-lite-v1:0", 
    inferenceConfig = {
        "maxTokens": 512,
        "temperature": 0.5
    },
    messages = [
        {
            "role": "user",
            "content": [{
                "text": prompt
            }]
        }
    ]
)

# Print the response output
print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "1564be99-9cd2-4a4f-a878-69a2b7853262",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 29 Jun 2026 15:27:31 GMT",
      "content-type": "application/json",
      "content-length": "319",
      "connection": "keep-alive",
      "x-amzn-requestid": "1564be99-9cd2-4a4f-a878-69a2b7853262"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "To introduce beginners to the basics of syntax and programming language structure in a simple and understandable way."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 14,
    "outputTokens": 20,
    "totalTokens": 34
  },
  "metrics": {
    "latencyMs": 493
  }
}


## ConverseStream API
Showing the entire output

In [14]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Invoke the model using converse
response = bedrock_runtime.converse_stream(
    modelId="us.amazon.nova-lite-v1:0", 
    inferenceConfig = {
        "maxTokens": 512,
        "temperature": 0.5
    },
    messages = [
        {
            "role": "user",
            "content": [{
                "text": prompt
            }]
        }
    ]
)

for event in response["stream"]:
    print(json.dumps(event, indent=2))

{
  "messageStart": {
    "role": "assistant"
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": "The"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " purpose of a '"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": "hello world' program"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " is to provide a simple and"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " basic"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " introduction"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " to"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " a programming"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text":

Showing only the text to see it as streamed

In [15]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in a text of at least 500 words"

# Invoke the model using converse
response = bedrock_runtime.converse_stream(
    modelId="amazon.nova-lite-v1:0", 
    inferenceConfig = {
        "maxTokens": 1024,
        "temperature": 0.5
    },
    messages = [
        {
            "role": "user",
            "content": [{
                "text": prompt
            }]
        }
    ]
)

for event in response["stream"]:
    if "contentBlockDelta" in event:
        chunk = event["contentBlockDelta"]
        sys.stdout.write(chunk["delta"]["text"])
        sys.stdout.flush()

The "Hello, World!" program holds a significant place in the history and culture of computer programming. It is one of the simplest and most basic programs that a new programmer can write, and yet it carries with it a wealth of importance and meaning. This seemingly trivial program serves as a foundational introduction to the world of coding, encapsulating numerous fundamental concepts and principles in a single, easily digestible package.

### The Genesis of "Hello, World!"

The "Hello, World!" program's origins can be traced back to the 1970s in the context of the Unix operating system. The program was first introduced in the seminal book "The C Programming Language" by Brian Kernighan and Dennis Ritchie. This book is often referred to as "The C Bible" and is credited with popularizing the C programming language. In this book, the authors used the "Hello, World!" program as the very first example to demonstrate the basic syntax and structure of the C programming language.

The simpli